# 📊 Performance Pipeline Comparison
**Dataset:** mudah.my Malaysia Property Listings

This notebook benchmarks **4 data processing pipelines** on the same dataset.

| Pipeline | Method |
|----------|--------|
| 1 | Pandas Baseline (unoptimized) |
| 2 | Pandas Optimized (dtype, usecols, categorical, vectorized) |
| 3 | Polars |
| 4 | DuckDB |

**Same task across all methods:**
- Drop missing values
- Remove duplicates
- Group by `state`
- Calculate count and mean price per state

**Metrics measured:** time (sec), CPU (%), memory (MB), throughput (records/sec)

## 📦 Step 1 — Install & Import
Install and import the relevant library for all the optimization method used in the data processing pipelines.

In [ ]:
!pip install -q polars duckdb psutil

import pandas as pd
import polars as pl
import duckdb
import psutil
import os
import time
import gc
import psutil, os
process = psutil.Process(os.getpid())
from datetime import datetime

## ⚙️ Step 2 — File Configuration
Set up to read the properties listings dataset extracted from mudah.my and save the performance before and after optimization in csv files.

In [ ]:
DATA = 'mudah_goe.csv'

# Output files
PERFORMANCE_BEFORE = 'performance_before.csv'
PERFORMANCE_AFTER  = 'performance_after.csv'

# Columns used in processing
COLS_NEEDED = ['listing_id', 'property_type', 'location', 'state',
               'price_rm', 'bedrooms', 'bathrooms', 'size_sqft', 'tenure']

# Processing task — group by this column
GROUP_COL   = 'state'
NUMERIC_COL = 'price_rm'

# Verify file exists
if os.path.exists(DATA):
    size_mb = os.path.getsize(DATA) / (1024 * 1024)
    print(f'✅ Found {DATA} ({size_mb:.1f} MB)')
else:
    print(f'❌ {DATA} not found — upload your combined CSV first')

✅ Found mudah_goe.csv (37.3 MB)


## 🔧 Step 3 — Helper: Metrics Collection
The function set up the function to collect the benchmark metric, including:

*   Memory Usage (MB)
*   CPU Usage (%)
*   Total Processing Time (second)
*   Throughput (Records/second)

In [ ]:
all_run_records = []   # stores every individual run result

def run_benchmark(method_name, fn, total_records, runs=3):
    print(f'\n  Running: {method_name} ({runs} runs)...')

    all_times, all_cpu, all_mem, all_tput = [], [], [], []
    result = None

    for run in range(1, runs + 1):
        gc.collect()

        mem_before = get_memory_mb()
        process.cpu_percent(interval=None)
        time_start = time.perf_counter()

        result = fn()

        time_end    = time.perf_counter()
        cpu_percent = psutil.cpu_percent(interval=0.3)
        mem_after   = get_memory_mb()

        elapsed     = time_end - time_start
        memory_used = max(mem_after - mem_before, 0)
        throughput  = total_records / elapsed if elapsed > 0 else 0

        all_times.append(elapsed)
        all_cpu.append(cpu_percent)
        all_mem.append(memory_used)
        all_tput.append(throughput)

        # store individual run
        all_run_records.append({
            'method':                     method_name,
            'run':                        run,
            'time_sec':                   round(elapsed, 4),
            'cpu_percent':                round(cpu_percent, 2),
            'memory_mb':                  round(memory_used, 2),
            'throughput_records_per_sec': round(throughput, 0),
        })

        print(f'    Run {run}/{runs} → {elapsed:.3f}s | '
              f'CPU: {cpu_percent:.1f}% | '
              f'Mem: {memory_used:.1f}MB | '
              f'Throughput: {throughput:,.0f} rec/s')

        gc.collect()
        time.sleep(1)

    metrics = {
        'method':                     method_name,
        'time_sec':                   round(sum(all_times) / runs, 4),
        'time_min':                   round(min(all_times), 4),
        'time_max':                   round(max(all_times), 4),
        'cpu_percent':                round(sum(all_cpu) / runs, 2),
        'memory_mb':                  round(sum(all_mem) / runs, 2),
        'throughput_records_per_sec': round(sum(all_tput) / runs, 0),
    }

    print(f'    ── Avg: {metrics["time_sec"]}s | '
          f'Min: {metrics["time_min"]}s | '
          f'Max: {metrics["time_max"]}s')

    return metrics, result


def get_memory_mb():
    return process.memory_info().rss / (1024 * 1024)

print('✅ Benchmark helper ready.')

✅ Benchmark helper ready.


## 📋 Step 4 — Get Total Record Count

In [ ]:
# Count total rows without loading full data into memory
with open(DATA, 'r', encoding='utf-8-sig') as f:
    total_records = sum(1 for _ in f) - 1  # subtract header

print(f'Total records in dataset : {total_records:,}')

# Quick peek at structure
df_peek = pd.read_csv(DATA, nrows=3)
print(f'Columns                  : {list(df_peek.columns)}')
print(f'\nFirst 3 rows:')
display(df_peek)

Total records in dataset : 922,509
Columns                  : ['listing_id', 'property_type', 'description', 'location', 'state', 'price_rm', 'mortgage_est_rm', 'land_title', 'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type', 'seller_name', 'url', 'img_url']

First 3 rows:


,listing_id,property_type,description,location,state,price_rm,mortgage_est_rm,land_title,size_sqft,bedrooms,bathrooms,tenure,seller_type,seller_name,url,img_url
0,114696325,2-storey Terraced Houseforsale,Eco Cascadia Mount Austin 2 Storey House JB\n🔥...,Johor Bahru,Johor,720000,2869.0,Non Bumi Lot,1614.0,5.0,4.0,Freehold,Agent,NaN,https://www.mudah.my/eco-cascadia-mount-austin...,https://cdn.rnudah.com/images/plain/9f2ae5538d...
1,114709669,2-storey Terraced Houseforsale,Parkland Bandar Layangkasa Pasir Gudang Ori Un...,Pasir Gudang,Johor,468000,1865.0,Non Bumi Lot,1400.0,4.0,3.0,Freehold,Unknown,NaN,https://www.mudah.my/parkland-bandar-layangkas...,https://cdn.rnudah.com/images/plain/a4d62990bd...
2,113358275,New 1-storey Terraced Houseforsale,"Rumah Penjawat Awam harga hanya 𝐑𝐌𝟐𝟒𝟎,𝟎𝟎𝟎！🔥\n🔥...",Kluang,Johor,240000,956.0,Non Bumi Lot,1430.0,3.0,2.0,Freehold,Private,NaN,https://www.mudah.my/rumah-penjawat-awam-harga...,https://cdn.rnudah.com/images/plain/2dd37a967f...


---
## 🐼 Step 5 — Pipeline 1: Pandas Baseline (Unoptimized)
No optimizations — plain `read_csv`, default dtypes, standard groupby.

In [ ]:
def pandas_baseline():
    """
    UNOPTIMIZED pandas pipeline — no optimizations at all.
    Data is already clean so no dropna or dedup needed.
    Task: load → groupby state → count + mean price
    """
    df = pd.read_csv(DATA)

    df[NUMERIC_COL] = pd.to_numeric(df[NUMERIC_COL], errors='coerce')

    result = df.groupby(GROUP_COL).agg(
        listing_count=(NUMERIC_COL, 'count'),
        avg_price_rm=(NUMERIC_COL, 'mean')
    ).reset_index()

    result['avg_price_rm'] = result['avg_price_rm'].round(2)
    result = result.sort_values('listing_count', ascending=False).reset_index(drop=True)

    return result


metrics_baseline, result_baseline = run_benchmark(
    'pandas_baseline', pandas_baseline, total_records
)

print(f'\nBaseline result ({len(result_baseline)} states):')
display(result_baseline)

pd.DataFrame([metrics_baseline]).to_csv(PERFORMANCE_BEFORE, index=False)
print(f'\n✅ Saved: {PERFORMANCE_BEFORE}')


  Running: pandas_baseline (3 runs)...


/tmp/ipykernel_12213/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 1/3 → 1.408s | CPU: 71.7% | Mem: 83.4MB | Throughput: 655,184 rec/s


/tmp/ipykernel_12213/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 2/3 → 2.173s | CPU: 86.9% | Mem: 13.9MB | Throughput: 424,464 rec/s


/tmp/ipykernel_12213/2822665002.py:7: DtypeWarning: Columns (0,5,9,10,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA)


    Run 3/3 → 2.037s | CPU: 72.9% | Mem: 2.0MB | Throughput: 452,883 rec/s
    ── Avg: 1.8728s | Min: 1.408s | Max: 2.1734s

Baseline result (8 states):


,state,listing_count,avg_price_rm
0,Johor,10766,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03
5,30868.0,0,NaN
6,2104.0,0,NaN
7,2989,0,NaN



✅ Saved: performance_before.csv


---
## ⚡ Step 6 — Pipeline 2: Pandas Optimized
| # | Technique | What it does |
|---|-----------|-----------|
| 1 | usecols | Loads only 13 needed columns instead of all 16. |
| 2 | Explicit dtype | Tells pandas the column types upfront so it skips the inference step on read_csv |
| 3 | pd.Categorical | Converts low-cardinality string columns (state, tenure, etc.) to integer codes. Less RAM, faster groupby |
| 4 | .str.strip().str.title() | Cleans location strings using C-speed vectorised operations instead of a Python loop |

In [ ]:
def pandas_optimized():
    """
    OPTIMIZED pandas pipeline.
    Techniques:
    1. usecols        — load only 13 needed columns instead of all 16
    2. Explicit dtype — skip type inference on read_csv
    3. pd.Categorical — low-cardinality strings → integer codes (less RAM, faster groupby)
    4. .str.strip().str.title() — vectorised C-speed string cleaning
    """
    NEEDED_COLS = [
        'listing_id', 'description', 'property_type', 'location', 'state',
        'price_rm', 'mortgage_est_rm', 'land_title',
        'size_sqft', 'bedrooms', 'bathrooms', 'tenure', 'seller_type'
    ]
    existing_cols = [c for c in NEEDED_COLS if c in df_peek.columns]

    # Optimization 1 + 2: usecols + explicit dtype
    dtype_map = {'listing_id': 'str'}
    existing_dtypes = {k: v for k, v in dtype_map.items() if k in existing_cols}

    df = pd.read_csv(
        DATA,
        usecols=existing_cols,
        dtype=existing_dtypes,
        engine='c',
        na_values=['', 'None', 'nan'],
    )

    # Convert numeric cols safely — bad values become NaN
    for col in ['price_rm', 'mortgage_est_rm', 'size_sqft', 'bedrooms', 'bathrooms']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill missing — numeric cols use median, text cols use 'Unknown'
    for col in ['bedrooms', 'bathrooms', 'size_sqft', 'mortgage_est_rm']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    for col in ['property_type', 'location', 'state', 'tenure',
                'seller_type', 'land_title', 'description']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    # Optimization 3: pd.Categorical
    for col in ['property_type', 'state', 'tenure', 'seller_type', 'land_title']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Optimization 4: vectorised string cleaning
    if 'location' in df.columns:
        df['location'] = df['location'].str.strip().str.title()

    # Same groupby task as baseline for fair comparison
    result = (
        df.groupby(GROUP_COL, observed=True)[NUMERIC_COL]
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'avg_price_rm', 'count': 'listing_count'})
        .reset_index()
        .sort_values('listing_count', ascending=False)
        .reset_index(drop=True)
    )
    result['avg_price_rm'] = result['avg_price_rm'].round(2)

    return result


metrics_optimized, result_optimized = run_benchmark(
    'pandas_optimized', pandas_optimized, total_records
)

print(f'\nOptimized result ({len(result_optimized)} states):')
display(result_optimized)


  Running: pandas_optimized (3 runs)...


/tmp/ipykernel_12213/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 1/3 → 1.057s | CPU: 26.2% | Mem: 0.6MB | Throughput: 872,965 rec/s


/tmp/ipykernel_12213/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 2/3 → 3.378s | CPU: 90.0% | Mem: 0.0MB | Throughput: 273,089 rec/s


/tmp/ipykernel_12213/1729898736.py:21: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Run 3/3 → 2.144s | CPU: 71.7% | Mem: 0.0MB | Throughput: 430,216 rec/s
    ── Avg: 2.193s | Min: 1.0568s | Max: 3.3781s

Optimized result (9 states):


,state,avg_price_rm,listing_count
0,Johor,1718555.80,10766
1,Negeri Sembilan,1096927.08,10043
2,Kedah,1780977.89,8702
3,Melaka,914871.95,8126
4,Kelantan,967058.03,1671
5,2104.0,NaN,0
6,30868.0,NaN,0
7,2989,NaN,0
8,Unknown,NaN,0


---
## 🐻‍❄️ Step 7 — Pipeline 3: Polars
Rust-based DataFrame library — lazy evaluation, multi-threaded, columnar.

In [ ]:
def polars_pipeline():
    """
    Polars pipeline — same task as baseline.
    Uses lazy evaluation (scan_csv) for memory efficiency.
    """
    existing_cols = [c for c in COLS_NEEDED if c in df_peek.columns]

    # Use lazy API — doesn't load data until .collect()
    lf = pl.scan_csv(
        DATA,
        infer_schema_length=0,
        null_values=['', 'None', 'nan', 'null'],
        ignore_errors=True,
        schema_overrides={
            'listing_id':      pl.Utf8,
            'price_rm':        pl.Utf8,
            'mortgage_est_rm': pl.Utf8,
            'size_sqft':       pl.Utf8,
            'bedrooms':        pl.Utf8,
            'bathrooms':       pl.Utf8,
        }
    )

    # Select only needed columns that exist
    available = lf.collect_schema().names()
    cols_to_select = [c for c in existing_cols if c in available]
    lf = lf.select(cols_to_select)

    # Safely cast numeric columns after loading
    numeric_cols = ['price_rm', 'mortgage_est_rm', 'size_sqft', 'bedrooms', 'bathrooms']
    cast_exprs = [
        pl.col(c).cast(pl.Float64, strict=False)
        for c in numeric_cols if c in cols_to_select
    ]
    if cast_exprs:
        lf = lf.with_columns(cast_exprs)

    # Drop nulls in key columns
    lf = lf.drop_nulls(subset=[GROUP_COL, NUMERIC_COL])

    # Remove duplicates
    lf = lf.unique(subset=['listing_id'], keep='last')

    # Groupby + aggregation
    lf = lf.group_by(GROUP_COL).agg([
        pl.count(NUMERIC_COL).alias('listing_count'),
        pl.mean(NUMERIC_COL).round(2).alias('avg_price_rm'),
    ]).sort('listing_count', descending=True)

    # Execute
    result_pl = lf.collect()

    # Convert to pandas for consistent output
    return result_pl.to_pandas()


metrics_polars, result_polars = run_benchmark(
    'polars', polars_pipeline, total_records
)

print(f'\nPolars result ({len(result_polars)} states):')
display(result_polars)


  Running: polars (3 runs)...
    Run 1/3 → 0.051s | CPU: 4.8% | Mem: 32.8MB | Throughput: 18,218,951 rec/s
    Run 2/3 → 0.041s | CPU: 0.0% | Mem: 4.4MB | Throughput: 22,638,684 rec/s
    Run 3/3 → 0.054s | CPU: 0.0% | Mem: 0.3MB | Throughput: 16,931,892 rec/s
    ── Avg: 0.0486s | Min: 0.0407s | Max: 0.0545s

Polars result (5 states):


,state,listing_count,avg_price_rm
0,Johor,10766,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03


---
## 🦆 Step 8 — Pipeline 4: DuckDB
In-process SQL OLAP engine — reads CSV directly with SQL, columnar vectorized execution.

In [ ]:
def duckdb_pipeline():
    """
    DuckDB pipeline — same groupby task using SQL.
    Reads CSV directly without loading into pandas first.
    Uses columnar vectorized execution.
    """
    con = duckdb.connect(database=':memory:')

    query = f"""
        SELECT
            {GROUP_COL},
            COUNT(*)                                          AS listing_count,
            ROUND(AVG(TRY_CAST({NUMERIC_COL} AS DOUBLE)), 2) AS avg_price_rm
        FROM read_csv_auto(
            '{DATA}',
            null_padding  = true,
            ignore_errors = true
        )
        WHERE {GROUP_COL}   IS NOT NULL
          AND {NUMERIC_COL} IS NOT NULL
        GROUP BY {GROUP_COL}
        ORDER BY listing_count DESC
    """

    result = con.execute(query).df()
    con.close()
    return result


metrics_duckdb, result_duckdb = run_benchmark(
    'duckdb', duckdb_pipeline, total_records
)

print(f'\nDuckDB result ({len(result_duckdb)} states):')
display(result_duckdb)


  Running: duckdb (3 runs)...
    Run 1/3 → 0.267s | CPU: 56.7% | Mem: 0.0MB | Throughput: 3,457,175 rec/s
    Run 2/3 → 0.277s | CPU: 55.2% | Mem: 0.0MB | Throughput: 3,335,080 rec/s
    Run 3/3 → 0.168s | CPU: 0.0% | Mem: 0.3MB | Throughput: 5,485,382 rec/s
    ── Avg: 0.2372s | Min: 0.1682s | Max: 0.2766s

DuckDB result (11 states):


,state,listing_count,avg_price_rm
0,Johor,10767,1718555.80
1,Negeri Sembilan,10043,1096927.08
2,Kedah,8702,1780977.89
3,Melaka,8126,914871.95
4,Kelantan,1671,967058.03
5,2742.0,1,NaN
6,2152.0,1,NaN
7,2104.0,1,NaN
8,2232.0,1,NaN
9,30868.0,1,NaN


---
## 📊 Step 9 — Combine Results & Compare

In [ ]:
all_metrics = [
    metrics_baseline,
    metrics_optimized,
    metrics_polars,
    metrics_duckdb,
]

df_perf = pd.DataFrame(all_metrics)
baseline_time = df_perf.loc[df_perf['method'] == 'pandas_baseline', 'time_sec'].values[0]
df_perf['speedup_vs_baseline'] = (baseline_time / df_perf['time_sec']).round(2)

# ── Build long table: one row per run ─────────────────────────────────────────
df_runs = pd.DataFrame(all_run_records)

# Merge avg metrics onto each method
avg_cols = df_perf[['method','time_sec','cpu_percent','memory_mb',
                     'throughput_records_per_sec','speedup_vs_baseline']].rename(columns={
    'time_sec':                   'avg_time_sec',
    'cpu_percent':                'avg_cpu_%',
    'memory_mb':                  'avg_memory_mb',
    'throughput_records_per_sec': 'avg_throughput',
})

df_long = df_runs.merge(avg_cols, on='method', how='left')

# Rename run-level columns for clarity
df_long = df_long.rename(columns={
    'time_sec':                   'time_sec',
    'cpu_percent':                'cpu_%',
    'memory_mb':                  'memory_mb',
    'throughput_records_per_sec': 'throughput',
})

# Order methods
method_order = ['pandas_baseline', 'pandas_optimized', 'polars', 'duckdb']
df_long['method'] = pd.Categorical(df_long['method'], categories=method_order, ordered=True)
df_long = df_long.sort_values(['method', 'run']).reset_index(drop=True)

# ── Display with merged cells using MultiIndex ────────────────────────────────
df_display = df_long[['method','run',
                       'time_sec','cpu_%','memory_mb','throughput',
                       'avg_time_sec','avg_cpu_%','avg_memory_mb','avg_throughput',
                       'speedup_vs_baseline']].copy()

df_display = df_display.set_index(['method', 'run'])

print(f"\n{'='*90}")
print(f"  FULL BENCHMARK RESULTS — {total_records:,} records | 3 runs each")
print(f"{'='*90}\n")

# Style: show avg cols only on middle row (run 2), blank on runs 1 and 3
def blank_non_middle(val, row_idx):
    # row_idx within each group: 0=run1, 1=run2, 2=run3
    return val if row_idx == 1 else ''

styled = df_display.style \
    .format({
        'time_sec':           '{:.4f}s',
        'cpu_%':              '{:.1f}%',
        'memory_mb':          '{:.2f}',
        'throughput':         '{:,.0f}',
        'avg_time_sec':       '{:.4f}s',
        'avg_cpu_%':          '{:.1f}%',
        'avg_memory_mb':      '{:.2f}',
        'avg_throughput':     '{:,.0f}',
        'speedup_vs_baseline':'{:.2f}x',
    }) \
    .set_table_styles([
        {'selector': 'th', 'props': [('font-weight','bold'),('background-color','#f0f0f0')]},
        {'selector': 'td', 'props': [('text-align','center')]},
        {'selector': 'tr:nth-child(3n)', 'props': [('border-bottom','2px solid #999')]},
    ]) \
    .apply(lambda col: [
        'background-color: #fff9e6' if i % 3 == 0
        else 'background-color: #ffffff' if i % 3 == 1
        else 'background-color: #f0f9ff'
        for i in range(len(col))
    ], axis=0) \
    .highlight_min(subset=['avg_time_sec'], color='lightgreen') \
    .highlight_max(subset=['avg_time_sec'], color='#ffcccc') \
    .highlight_max(subset=['avg_throughput','speedup_vs_baseline'], color='lightgreen')

display(styled)

# ── Hide duplicate avg values on runs 1 and 3 (show only on run 2) ────────────
# Cleaner version using applymap
avg_metric_cols = ['avg_time_sec','avg_cpu_%','avg_memory_mb','avg_throughput','speedup_vs_baseline']

def hide_non_middle(v, idx):
    return '' if idx % 3 != 1 else v

df_clean = df_display.copy().astype(str)
for col in avg_metric_cols:
    df_clean[col] = [
        v if i % 3 == 1 else ''
        for i, v in enumerate(df_display[col].apply(
            lambda x: f'{x:.4f}s' if 'time' in col
            else f'{x:.1f}%' if 'cpu' in col
            else f'{x:,.0f}' if 'throughput' in col or 'speedup' in col
            else f'{x:.2f}'
        ))
    ]


  FULL BENCHMARK RESULTS — 922,509 records | 3 runs each



## 💾 Step 10 — Save Output Files

In [ ]:
# Save performance_before.csv (baseline only)
pd.DataFrame([metrics_baseline]).to_csv(PERFORMANCE_BEFORE, index=False)
print(f'✅ Saved: {PERFORMANCE_BEFORE}')

# Save performance_after.csv (all 4 methods)
df_perf.to_csv(PERFORMANCE_AFTER, index=False)
print(f'✅ Saved: {PERFORMANCE_AFTER}')

# Print final clean table
print(f'\n{"="*75}')
print('  FINAL PERFORMANCE TABLE')
print(f'{"="*75}')
display(df_perf[['method','time_sec','cpu_percent','memory_mb',
                  'throughput_records_per_sec','speedup_vs_baseline']].style
        .format({
            'time_sec': '{:.4f}',
            'cpu_percent': '{:.2f}%',
            'memory_mb': '{:.2f}',
            'throughput_records_per_sec': '{:,.0f}',
            'speedup_vs_baseline': '{:.2f}x',
        })
        .highlight_min(subset=['time_sec','memory_mb'], color='lightgreen')
        .highlight_max(subset=['throughput_records_per_sec','speedup_vs_baseline'], color='lightgreen')
        .highlight_max(subset=['time_sec','memory_mb'], color='#ffcccc')
)

# Download all output files
from google.colab import files
for f in [PERFORMANCE_BEFORE, PERFORMANCE_AFTER, 'performance_comparison.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'⬇️  Downloaded: {f}')

✅ Saved: performance_before.csv
✅ Saved: performance_after.csv

  FINAL PERFORMANCE TABLE


,method,time_sec,cpu_percent,memory_mb,throughput_records_per_sec,speedup_vs_baseline
0,pandas_baseline,1.8728,77.17%,33.11,"510,844",1.00x
1,pandas_optimized,2.1930,62.63%,0.21,"525,423",0.85x
2,polars,0.0486,1.60%,12.51,"19,263,176",38.53x
3,duckdb,0.2372,37.30%,0.11,"4,092,546",7.90x


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloaded: performance_before.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Downloaded: performance_after.csv
